In [1]:
import pandas as pd

In [2]:
df =  pd.read_csv('../data/features/featured_data.csv')

In [3]:
df.dtypes

product                            str
category                           str
issue_description                  str
priority                           str
channel                            str
region                             str
customer_age                     int64
customer_gender                    str
subscription_type                  str
customer_tenure_months           int64
previous_tickets                 int64
customer_satisfaction_score      int64
first_response_time_hours      float64
escalated                          str
sla_breached                       str
operating_system                   str
browser                            str
payment_method                     str
language                           str
preferred_contact_time             str
customer_segment                   str
day_of_week                      int64
month                            int64
ticket_rate                    float64
dtype: object

In [4]:
df.select_dtypes(include=['int64', 'float64']).head(10)

,customer_age,customer_tenure_months,previous_tickets,customer_satisfaction_score,first_response_time_hours,day_of_week,month,ticket_rate
0,31,36,6,5,30.50,2,5,0.154151
1,23,54,20,5,63.81,5,1,0.315081
2,47,60,20,5,16.20,2,11,0.287682
3,65,58,18,4,26.38,3,3,0.270290
4,45,1,8,5,54.98,4,8,2.197225
5,23,53,2,5,55.26,4,9,0.037041
6,55,38,7,1,5.58,6,2,0.169076
7,44,44,20,1,4.83,0,8,0.374693
8,44,14,12,1,12.27,2,6,0.619039
9,21,55,5,1,43.05,3,5,0.087011


In [5]:
# Targets
y_priority = df['priority']
y_sla = df['sla_breached']
y_satisfaction = df['customer_satisfaction_score']

# Features (drop all 3 targets)
X = df.drop(columns=['priority', 'sla_breached', 'customer_satisfaction_score'])

In [6]:
X.shape

(200000, 21)

In [7]:
X_priority = X.drop(columns=['first_response_time_hours', 'escalated'])

In [8]:
X_sla = X.copy()

In [9]:
X_satisfaction = X.copy()

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [11]:
# Priority Classification
X_priority_train_val, X_priority_test, y_priority_train_val, y_priority_test = train_test_split(
    X_priority, y_priority, test_size=0.15, random_state=42
)
X_priority_train, X_priority_val, y_priority_train, y_priority_val = train_test_split(
    X_priority_train_val, y_priority_train_val, test_size=0.176, random_state=42
)

# SLA Breach
X_sla_train_val, X_sla_test, y_sla_train_val, y_sla_test = train_test_split(
    X_sla, y_sla, test_size=0.15, random_state=42
)
X_sla_train, X_sla_val, y_sla_train, y_sla_val = train_test_split(
    X_sla_train_val, y_sla_train_val, test_size=0.176, random_state=42
)

# Satisfaction
X_satisfaction_train_val, X_satisfaction_test, y_satisfaction_train_val, y_satisfaction_test = train_test_split(
    X_satisfaction, y_satisfaction, test_size=0.15, random_state=42
)
X_satisfaction_train, X_satisfaction_val, y_satisfaction_train, y_satisfaction_val = train_test_split(
    X_satisfaction_train_val, y_satisfaction_train_val, test_size=0.176, random_state=42
)

print(X_priority_train.shape, X_priority_val.shape, X_priority_test.shape)
print(X_sla_train.shape, X_sla_val.shape, X_sla_test.shape)
print(X_satisfaction_train.shape, X_satisfaction_val.shape, X_satisfaction_test.shape)

(140080, 19) (29920, 19) (30000, 19)
(140080, 21) (29920, 21) (30000, 21)
(140080, 21) (29920, 21) (30000, 21)


In [12]:
le_priority = LabelEncoder()
y_priority_train = le_priority.fit_transform(y_priority_train)
y_priority_val = le_priority.transform(y_priority_val)
y_priority_test = le_priority.transform(y_priority_test)

le_sla = LabelEncoder()
y_sla_train = le_sla.fit_transform(y_sla_train)
y_sla_val = le_sla.transform(y_sla_val)
y_sla_test = le_sla.transform(y_sla_test)

print(le_priority.classes_)
print(le_sla.classes_)

['high' 'low' 'medium' 'urgent']
['no' 'yes']


In [13]:
# Remove issue_description from cat_cols
cat_cols_priority = [col for col in X_priority_train.select_dtypes(include='object').columns if col != 'issue_description']
cat_cols_sla_and_satisfaction = [col for col in X_sla_train.select_dtypes(include='object').columns if col != 'issue_description']
print(cat_cols_priority)
print(cat_cols_sla_and_satisfaction)

['product', 'category', 'channel', 'region', 'customer_gender', 'subscription_type', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'customer_segment']
['product', 'category', 'channel', 'region', 'customer_gender', 'subscription_type', 'escalated', 'operating_system', 'browser', 'payment_method', 'language', 'preferred_contact_time', 'customer_segment']


/tmp/ipykernel_33644/181688912.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_priority = [col for col in X_priority_train.select_dtypes(include='object').columns if col != 'issue_description']
/tmp/ipykernel_33644/181688912.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [14]:
def encode_cats(X, ohe, cat_cols):
    encoded = ohe.transform(X[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols), index=X.index)
    return pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)


In [15]:
# Priority OHE
ohe_priority = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_priority.fit(X_priority_train[cat_cols_priority])
X_priority_train = encode_cats(X_priority_train, ohe_priority, cat_cols_priority)
X_priority_val = encode_cats(X_priority_val, ohe_priority, cat_cols_priority)
X_priority_test = encode_cats(X_priority_test, ohe_priority, cat_cols_priority)

# SLA and Satisfaction OHE — same encoder fitted once
ohe_sla_satisfaction = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_sla_satisfaction.fit(X_sla_train[cat_cols_sla_and_satisfaction])

X_sla_train = encode_cats(X_sla_train, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)
X_sla_val = encode_cats(X_sla_val, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)
X_sla_test = encode_cats(X_sla_test, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)

X_satisfaction_train = encode_cats(X_satisfaction_train, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)
X_satisfaction_val = encode_cats(X_satisfaction_val, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)
X_satisfaction_test = encode_cats(X_satisfaction_test, ohe_sla_satisfaction, cat_cols_sla_and_satisfaction)

print(X_priority_train.shape)
print(X_sla_train.shape)
print(X_satisfaction_train.shape)

(140080, 73)
(140080, 76)
(140080, 76)


In [16]:
num_cols_priority = ['customer_age', 'customer_tenure_months', 'previous_tickets', 
                     'day_of_week', 'month', 'ticket_rate']

num_cols_sla_satisfaction = ['customer_age', 'customer_tenure_months', 'previous_tickets',
                              'first_response_time_hours', 'day_of_week', 'month', 'ticket_rate']

In [17]:
# Priority scaler
scaler_priority = StandardScaler()
X_priority_train[num_cols_priority] = scaler_priority.fit_transform(X_priority_train[num_cols_priority])
X_priority_val[num_cols_priority] = scaler_priority.transform(X_priority_val[num_cols_priority])
X_priority_test[num_cols_priority] = scaler_priority.transform(X_priority_test[num_cols_priority])

# SLA scaler
scaler_sla = StandardScaler()
X_sla_train[num_cols_sla_satisfaction] = scaler_sla.fit_transform(X_sla_train[num_cols_sla_satisfaction])
X_sla_val[num_cols_sla_satisfaction] = scaler_sla.transform(X_sla_val[num_cols_sla_satisfaction])
X_sla_test[num_cols_sla_satisfaction] = scaler_sla.transform(X_sla_test[num_cols_sla_satisfaction])

# Satisfaction scaler
scaler_satisfaction = StandardScaler()
X_satisfaction_train[num_cols_sla_satisfaction] = scaler_satisfaction.fit_transform(X_satisfaction_train[num_cols_sla_satisfaction])
X_satisfaction_val[num_cols_sla_satisfaction] = scaler_satisfaction.transform(X_satisfaction_val[num_cols_sla_satisfaction])
X_satisfaction_test[num_cols_sla_satisfaction] = scaler_satisfaction.transform(X_satisfaction_test[num_cols_sla_satisfaction])

print(X_priority_train[num_cols_priority].describe().round(2))

       customer_age  customer_tenure_months  previous_tickets  day_of_week  \
count     140080.00               140080.00         140080.00     140080.0   
mean           0.00                    0.00              0.00          0.0   
std            1.00                    1.00              1.00          1.0   
min           -1.70                   -1.70             -1.65         -1.5   
25%           -0.86                   -0.89             -0.82         -1.0   
50%           -0.03                   -0.02              0.00          0.0   
75%            0.87                    0.84              0.83          1.0   
max            1.70                    1.71              1.65          1.5   

           month  ticket_rate  
count  140080.00    140080.00  
mean       -0.00         0.00  
std         1.00         1.00  
min        -1.60        -0.92  
25%        -0.73        -0.60  
50%         0.14        -0.29  
75%         1.01         0.18  
max         1.59         5.75  


In [18]:
print(X_sla_train[num_cols_sla_satisfaction].describe().round(2))

       customer_age  customer_tenure_months  previous_tickets  \
count     140080.00               140080.00         140080.00   
mean           0.00                    0.00              0.00   
std            1.00                    1.00              1.00   
min           -1.70                   -1.70             -1.65   
25%           -0.86                   -0.89             -0.82   
50%           -0.03                   -0.02              0.00   
75%            0.87                    0.84              0.83   
max            1.70                    1.71              1.65   

       first_response_time_hours  day_of_week      month  ticket_rate  
count                  140080.00     140080.0  140080.00    140080.00  
mean                       -0.00          0.0      -0.00         0.00  
std                         1.00          1.0       1.00         1.00  
min                        -1.73         -1.5      -1.60        -0.92  
25%                        -0.86         -1.0      -0.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50, stop_words='english')

# Fit on priority train only
tfidf.fit(X_priority_train['issue_description'])

# Transform all splits for priority
X_priority_train_text = tfidf.transform(X_priority_train['issue_description'])
X_priority_val_text = tfidf.transform(X_priority_val['issue_description'])
X_priority_test_text = tfidf.transform(X_priority_test['issue_description'])

print(X_priority_train_text.shape)

(140080, 50)


In [20]:
X_sla_train_text = tfidf.transform(X_sla_train['issue_description'].astype(str))
X_sla_val_text = tfidf.transform(X_sla_val['issue_description'].astype(str))
X_sla_test_text = tfidf.transform(X_sla_test['issue_description'].astype(str))

X_satisfaction_train_text = tfidf.transform(X_satisfaction_train['issue_description'].astype(str))
X_satisfaction_val_text = tfidf.transform(X_satisfaction_val['issue_description'].astype(str))
X_satisfaction_test_text = tfidf.transform(X_satisfaction_test['issue_description'].astype(str))

In [21]:
import numpy as np
from scipy.sparse import hstack, csr_matrix

# Drop issue_description from all splits
X_priority_train = X_priority_train.drop(columns=['issue_description'])
X_priority_val = X_priority_val.drop(columns=['issue_description'])
X_priority_test = X_priority_test.drop(columns=['issue_description'])

# Combine structured + text features
X_priority_train_final = hstack([csr_matrix(X_priority_train.values), X_priority_train_text])
X_priority_val_final = hstack([csr_matrix(X_priority_val.values), X_priority_val_text])
X_priority_test_final = hstack([csr_matrix(X_priority_test.values), X_priority_test_text])

print(X_priority_train_final.shape)

(140080, 122)


In [22]:
X_sla_train = X_sla_train.drop(columns=['issue_description'])
X_sla_val = X_sla_val.drop(columns=['issue_description'])
X_sla_test = X_sla_test.drop(columns=['issue_description'])

X_sla_train_final = hstack([csr_matrix(X_sla_train.values), X_sla_train_text])
X_sla_val_final = hstack([csr_matrix(X_sla_val.values), X_sla_val_text])
X_sla_test_final = hstack([csr_matrix(X_sla_test.values), X_sla_test_text])

X_satisfaction_train = X_satisfaction_train.drop(columns=['issue_description'])
X_satisfaction_val = X_satisfaction_val.drop(columns=['issue_description'])
X_satisfaction_test = X_satisfaction_test.drop(columns=['issue_description'])

X_satisfaction_train_final = hstack([csr_matrix(X_satisfaction_train.values), X_satisfaction_train_text])
X_satisfaction_val_final = hstack([csr_matrix(X_satisfaction_val.values), X_satisfaction_val_text])
X_satisfaction_test_final = hstack([csr_matrix(X_satisfaction_test.values), X_satisfaction_test_text])

print(X_sla_train_final.shape)
print(X_satisfaction_train_final.shape)

(140080, 125)
(140080, 125)


In [25]:
import joblib
import numpy as np
from scipy.sparse import save_npz

# Save sparse matrices
save_npz('../data/features/X_priority_train.npz', X_priority_train_final)
save_npz('../data/features/X_priority_val.npz', X_priority_val_final)
save_npz('../data/features/X_priority_test.npz', X_priority_test_final)

save_npz('../data/features/X_sla_train.npz', X_sla_train_final)
save_npz('../data/features/X_sla_val.npz', X_sla_val_final)
save_npz('../data/features/X_sla_test.npz', X_sla_test_final)

save_npz('../data/features/X_satisfaction_train.npz', X_satisfaction_train_final)
save_npz('../data/features/X_satisfaction_val.npz', X_satisfaction_val_final)
save_npz('../data/features/X_satisfaction_test.npz', X_satisfaction_test_final)

# Save target arrays
np.save('../data/features/y_priority_train.npy', y_priority_train)
np.save('../data/features/y_priority_val.npy', y_priority_val)
np.save('../data/features/y_priority_test.npy', y_priority_test)

np.save('../data/features/y_sla_train.npy', y_sla_train)
np.save('../data/features/y_sla_val.npy', y_sla_val)
np.save('../data/features/y_sla_test.npy', y_sla_test)

np.save('../data/features/y_satisfaction_train.npy', y_satisfaction_train)
np.save('../data/features/y_satisfaction_val.npy', y_satisfaction_val)
np.save('../data/features/y_satisfaction_test.npy', y_satisfaction_test)

# Save preprocessing artifacts
joblib.dump(ohe_priority, '../saved_models/ohe_priority.pkl')
joblib.dump(ohe_sla_satisfaction, '../saved_models/ohe_sla_satisfaction.pkl')
joblib.dump(scaler_priority, '../saved_models/scaler_priority.pkl')
joblib.dump(scaler_sla, '../saved_models/scaler_sla.pkl')
joblib.dump(scaler_satisfaction, '../saved_models/scaler_satisfaction.pkl')
joblib.dump(tfidf, '../saved_models/tfidf.pkl')
joblib.dump(le_priority, '../saved_models/le_priority.pkl')
joblib.dump(le_sla, '../saved_models/le_sla.pkl')

['../saved_models/le_sla.pkl']